# 12-1절 연습 문제 풀이

이 노트북은 12-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch12/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 12장 공통 - transformers 라이브러리
# 주의: 이 장의 예제는 모델 내려받기와 GPU가 필요하다.
#       CPU 환경이나 네트워크가 없는 환경에서는 실행되지 않을 수 있다.
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import transformers
    print(f'transformers {transformers.__version__}')
except ImportError:
    print('알림: pip install transformers 로 라이브러리를 먼저 설치한다.')

MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

## 연습 12-1

Bllossom-3B 모델 대신 관심 있는 다른 한국어 LLM을 허깅페이스 허브에서 골라 [코드 12-2], [코드 12-7]을 그대로 적용해 답변을 받아 보자. 모델마다 설정이 다를 수 있으므로 모델 카드의 내용을 꼭 확인하자.

In [ ]:
# 다른 한국어 LLM으로 바꿔 실행한다. (모델 카드의 설정을 반드시 확인)
ALT_MODEL = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'   # 예시 - 원하는 모델로 교체

tokenizer = AutoTokenizer.from_pretrained(ALT_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    ALT_MODEL, torch_dtype=torch.float16, device_map='auto')

messages = [{'role': 'user', 'content': '딥러닝을 한 문장으로 설명해 줘.'}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                       return_tensors='pt', return_dict=True).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128, do_sample=True,
                         temperature=0.7, top_p=0.9)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:],
                       skip_special_tokens=True))

모델을 바꿀 때 확인할 것은 세 가지다. ① **대화틀**(`apply_chat_template` 지원 여부와 형식), ② **자료형**(FP16 지원 여부), ③ **크기**(VRAM에 맞는지). 모델 카드에 권장 설정이 적혀 있으므로 반드시 확인한다.

## 연습 12-2

[코드 12-7]에서 0.2~1.5 사이의 여러 온도값으로 같은 질문에 답변을 여러 번 생성해 보자. 온도 샘플링이 어떤 차이를 만드는지 직접 관찰해 보자.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16,
                                             device_map='auto')
messages = [{'role': 'user', 'content': '가을을 주제로 짧은 시를 써 줘.'}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                       return_tensors='pt', return_dict=True).to(model.device)
for temp in (0.2, 0.7, 1.0, 1.5):
    print(f'=== 온도 {temp} ===')
    out = model.generate(**inputs, max_new_tokens=80, do_sample=True,
                         temperature=temp, top_p=0.95)
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:],
                           skip_special_tokens=True), '\n')

온도가 낮으면(0.2) 확률이 높은 토큰만 골라 **안전하지만 단조로운** 문장이 나오고, 여러 번 실행해도 결과가 비슷하다. 온도가 높으면(1.5) 확률이 낮은 토큰도 선택되어 **다양하지만 문맥이 어긋나거나 문법이 무너지기** 쉽다. 창작에는 0.7~1.0, 사실 전달에는 0.2~0.5가 무난하다.

## 연습 12-3

[도전 문제] 허깅페이스 허브에 등록된 여러 한국어 지원 LLM을 각자의 실습 환경에서 디스크 사용량, 메모리 사용량 등을 측정하며 테스트해 보자. 측정 방법은 운영 체제에 따라 다르니 각자 환경에 맞는 방법을 찾아 적용한다. 파라미터 수가 적은 모델부터 점차 큰 모델로 늘려 가며 내 실습 환경의 한계도 확인해 보자. 메모리 부족으로 모델을 불러올 수 없는 상황도 발생할 수 있다.

In [ ]:
import os, time
def measure(model_name, prompt='안녕하세요'):
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_name)
    m = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                             device_map='auto')
    load_time = time.time() - t0
    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024 ** 3
    else:
        vram = float('nan')
    n_param = sum(p.numel() for p in m.parameters())
    print(f'{model_name}')
    print(f'  파라미터 {n_param / 1e9:.2f}B / 불러오기 {load_time:.1f}초 / '
          f'VRAM {vram:.2f}GB')
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for name in [MODEL_NAME]:      # 관심 있는 모델을 목록에 추가한다
    measure(name)

VRAM 사용량은 대략 **파라미터 수 × 자료형 바이트 수**다. FP16이면 3B 모델이 약 6GB를 쓴다. 디스크 사용량은 허깅페이스 캐시(`~/.cache/huggingface`)에서 확인할 수 있다. 12-3절의 양자화를 적용하면 이 값을 크게 줄일 수 있다.

## 연습 12-4

[도전 문제] Bllossom-3B 모델로 대화를 이어 나가는 간단한 채팅 서비스 프로그램을 작성해 보자. 이전까지의 대화를 맥락으로 다음 답변을 생성하도록 구현해야 한다.

In [ ]:
# 대화 맥락을 유지하는 간단한 채팅 루프
def chat_once(model, tokenizer, history, user_input, max_new_tokens=200):
    history = history + [{'role': 'user', 'content': user_input}]
    inputs = tokenizer.apply_chat_template(history, add_generation_prompt=True,
                                           return_tensors='pt',
                                           return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:],
                              skip_special_tokens=True).strip()
    return history + [{'role': 'assistant', 'content': answer}], answer

history = []
for user_input in ['파이토치가 뭐야?', '방금 말한 것을 초등학생에게 설명하면?']:
    history, answer = chat_once(model, tokenizer, history, user_input)
    print(f'사용자: {user_input}\n모델: {answer}\n')

핵심은 **이전 대화를 `history` 리스트에 계속 누적**해 매번 통째로 넣는 것이다. LLM은 상태를 기억하지 못하므로 맥락을 프롬프트로 다시 제공해야 한다.

대화가 길어지면 컨텍스트 길이를 넘으므로, 오래된 대화를 자르거나 요약해 넣는 처리가 필요하다.